In [27]:
# Example plotting multiple values
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import SimpleITK as sitk
import pandas as pd
import os
import sys
from torchmetrics.image.fid import FrechetInceptionDistance
import copy

# sys.path.append("/mnt/raid/C1_ML_Analysis/source/autoencoder/src")
sys.path.append("/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl/")
sys.path.append("/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl/nets")

from nets import cut
from loaders import ultrasound_dataset as usd
import monai

import plotly.express as px
import plotly.graph_objects as go


In [28]:
def toint(x):
    return (x*255).clamp(0, 255).to(torch.uint8)

In [29]:
def _to_uint8_rgb(x):
    """(N,1,H,W) or (N,3,H,W) float in [0,1], numpy or torch -> uint8 (N,3,H,W)."""
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x)
    x = x.float()
    if x.shape[1] == 1:
        x = x.repeat(1, 3, 1, 1)
    return (x * 255).clamp(0, 255).to(torch.uint8)


def _update_batched(metric, data, real, batch_size=128):
    """Push images through the FID inception net in batches to avoid OOM."""
    for i in range(0, data.shape[0], batch_size):
        metric.update(_to_uint8_rgb(data[i:i + batch_size]).cuda(), real=real)


def compute_fid_all_types(all_manufacturers, all_manufacturers_fake, all_target,
                          model=None, batch_size=64, feature=2048):
    """FID of each manufacturer's real source images and its CUT 'fake' images
    against the (single) target domain. Lower fake-FID => better translation.

    The target's Inception features are computed once and reused for every
    comparison (deep-copying the metric state), so the expensive 2048-d target
    pass only runs a single time.

    Returns a tidy DataFrame: columns [manufacturer, type ('real'|'fake'), fid].
    `model` is unused (fakes are precomputed in all_manufacturers_fake) but kept
    in the signature for compatibility.
    """
    # The target reference distribution (e.g. 'Sonosite, Inc.' / Leltek).
    target_key = next(iter(all_target))
    target = all_target[target_key]

    # Compute the target (real) features ONCE.
    base = FrechetInceptionDistance(feature=feature).cuda()
    _update_batched(base, target, real=True, batch_size=batch_size)

    def _fid_against_target(data):
        metric = copy.deepcopy(base)          # reuse cached target real-features
        _update_batched(metric, data, real=False, batch_size=batch_size)
        return metric.compute().item()

    rows = []
    for manufacturer in all_manufacturers:
        rows.append({"manufacturer": manufacturer, "type": "real",
                     "fid": _fid_against_target(all_manufacturers[manufacturer])})
        rows.append({"manufacturer": manufacturer, "type": "fake",
                     "fid": _fid_against_target(all_manufacturers_fake[manufacturer])})

    return pd.DataFrame(rows)

In [30]:
def read_all(dl):
    dfs = dl.dataset.dfs

    all_manufacturers = {}
    all_target = {}

    for idx, row in dfs[0].iterrows():
        file_path = row["file_path"]
        manufacturer = row["manufacturer"]
        if all_manufacturers.get(manufacturer, None) is None:
            all_manufacturers[manufacturer] = []

        img = sitk.ReadImage(file_path)
        img_t = torch.from_numpy(sitk.GetArrayFromImage(img))

        if (img.GetNumberOfComponentsPerPixel() == 1):
            img_t = img_t.unsqueeze(-1)

        img_t = img_t[:,:,:,0:1].permute(0,3,1,2).to(torch.float32)/255.0

        all_manufacturers[manufacturer].append(img_t)

    for mk in all_manufacturers.keys():
        all_manufacturers[mk] = torch.cat(all_manufacturers[mk], dim=0)

    for idx, row in dfs[1].iterrows():
        file_path = row["file_path"]
        manufacturer = row["manufacturer"]
        if all_target.get(manufacturer, None) is None:
            all_target[manufacturer] = []

        img = sitk.ReadImage(file_path)
        img_t = torch.from_numpy(sitk.GetArrayFromImage(img)).to(torch.float32)/255.0

        if (img.GetNumberOfComponentsPerPixel() == 1):
            img_t = img_t.unsqueeze(-1)

        img_t = img_t[:,:,:,0:1].permute(0,3,1,2)

        all_target[manufacturer].append(img_t)

    for mk in all_target.keys():
        all_target[mk] = torch.cat(all_target[mk], dim=0)

    return all_manufacturers, all_target



In [31]:
import numpy as np
import io
from PIL import Image
import ipywidgets as widgets

def _frame_to_png_bytes(frame2d: np.ndarray) -> bytes:
    """Convert a (C,H,W) frame (any numeric dtype) to PNG bytes for ipywidgets.Image."""
    f = np.asarray(frame2d)
    # Handle NaNs/infs safely
    f = np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0)

    # Normalize to uint8
    fmin = float(f.min())
    fmax = float(f.max())
    if fmax > fmin:
        u8 = ((f - fmin) / (fmax - fmin) * 255.0).astype(np.uint8)
    else:
        u8 = np.zeros_like(f, dtype=np.uint8)

    img = Image.fromarray(u8, mode="RGB")  # RGB
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def _to_numpy(seq):
    """torch tensor / numpy / list of frames -> (T,H,W) or (T,H,W,C)."""
    if hasattr(seq, "detach"):
        seq = seq.detach().cpu().numpy()
    seq = np.squeeze(np.asarray(seq))
    # (T,C,H,W) -> (T,H,W,C)
    if seq.ndim == 4 and seq.shape[1] in (1, 3) and seq.shape[-1] not in (1, 3):
        seq = np.squeeze(np.transpose(seq, (0, 2, 3, 1)))
    return seq


def _to_THWC(seq):
    """torch/np/list -> (T,H,W,C=3) so _frame_to_png_bytes' RGB path works."""
    if hasattr(seq, "detach"):
        seq = seq.detach().cpu().numpy()
    seq = np.squeeze(np.asarray(seq))
    if seq.ndim == 4 and seq.shape[1] in (1, 3) and seq.shape[-1] not in (1, 3):
        seq = np.transpose(seq, (0, 2, 3, 1))           # (T,C,H,W)->(T,H,W,C)
    if seq.ndim == 3:
        seq = np.repeat(seq[..., None], 3, axis=-1)      # gray (T,H,W)->(T,H,W,3)
    if seq.ndim == 4 and seq.shape[-1] == 1:
        seq = np.repeat(seq, 3, axis=-1)
    return seq


def visualize_sequences(seq1, seq2, titles=("Sequence 1", "Sequence 2"),
                        *, fps=10.0, width="500px"):
    """Two image sequences side by side, single Play+slider driving both.
    Uses ipywidgets.Image (raw PNG bytes) — no plotly, no data URIs."""
    a, b = _to_THWC(seq1), _to_THWC(seq2)
    n = min(len(a), len(b))
    if len(a) != len(b):
        print(f"[viz] lengths differ ({len(a)} vs {len(b)}); showing first {n}")

    slider = widgets.IntSlider(value=0, min=0, max=n - 1, step=1, description="Frame",
                               continuous_update=True)
    play = widgets.Play(value=0, min=0, max=n - 1, step=1,
                        interval=int(1000 / max(fps, 1e-6)))
    widgets.jslink((play, "value"), (slider, "value"))

    img_a = widgets.Image(value=_frame_to_png_bytes(a[0]), format="png")
    img_b = widgets.Image(value=_frame_to_png_bytes(b[0]), format="png")
    img_a.layout.width = img_b.layout.width = width

    def _update(i):
        img_a.value = _frame_to_png_bytes(a[i])
        img_b.value = _frame_to_png_bytes(b[i])

    slider.observe(lambda ch: _update(ch["new"]), names="value")

    panel_a = widgets.VBox([widgets.HTML(f"<b>{titles[0]}</b>"), img_a])
    panel_b = widgets.VBox([widgets.HTML(f"<b>{titles[1]}</b>"), img_b])
    return widgets.VBox([widgets.HBox([play, slider]),
                         widgets.HBox([panel_a, panel_b])])



# Fréchet Inception Distance (FID)

The **Fréchet Inception Distance (FID)** is a metric used to evaluate the quality of images generated by generative models such as GANs. It compares the **distribution of generated images** to the **distribution of real images** using features extracted from the **Inception v3** neural network.

## Key Concepts

- FID computes the **Fréchet distance** (also known as the Wasserstein-2 distance) between two multivariate Gaussians:
  - One fitted to the **real image features**
  - One fitted to the **generated image features**

- The formula for FID:
  
  $$
  \text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})
  $$

  where:
  - $(\mu_r, \Sigma_r)$: mean and covariance of real images
  - $(\mu_g, \Sigma_g)$: mean and covariance of generated images

## Interpretation

- **Lower FID scores** indicate that the generated images are more similar to the real images.
- FID is preferred over earlier metrics (like Inception Score) because it **captures both the quality and diversity** of generated images.

## Applications

- Commonly used in research to benchmark generative models like **GANs**.
- Used in various image synthesis tasks, such as super-resolution, style transfer, and image generation.


In [ ]:

# model_fn = "/mnt/GWH/Groups/FAMLI/Shared/C1_ML_Analysis/train_output/Cut/allvslast/allvsleltek/v3.0/trial_2/epoch=49-val_loss_g_nce=3.56.ckpt"

model_fn = "/mnt/GWH/Groups/FAMLI/Shared/C1_ML_Analysis/train_output/Cut/allvslast/allvsleltek/v5.0/trial_2/epoch=96-val_loss_g_nce=3.45.ckpt"
# model_fn = "/mnt/GWH/Groups/FAMLI/Shared/C1_ML_Analysis/train_output/Cut/allvslast/allvsleltek/v5.0/trial_5/epoch=6-val_loss_g_nce=3.22.ckpt"
model = cut.CutG.load_from_checkpoint(model_fn).eval().cuda()

DATAMODULE = getattr(usd, model.hparams.data_module)
data = DATAMODULE(**model.hparams)

data.setup()
dl = data.val_dataloader()






KeyboardInterrupt: 

In [ ]:
all_manufacturers, all_target = read_all(dl)

In [ ]:
all_manufacturers_fake = {}
for manufacturer in all_manufacturers:
    all_manufacturers_fake[manufacturer] = []
    with torch.no_grad():
        splits = int(all_manufacturers[manufacturer].shape[0]/128)
        for s in torch.split(all_manufacturers[manufacturer], splits):
            s = s.cuda()
            s = model(s)
            all_manufacturers_fake[manufacturer].append(s.cpu())
        all_manufacturers_fake[manufacturer] = torch.cat(all_manufacturers_fake[manufacturer], dim=0).numpy()

In [ ]:
manufacturers = list(all_manufacturers.keys())

In [ ]:
manufacturer = manufacturers[0]
samples = 1000

idx = torch.randperm(all_manufacturers[manufacturer].shape[0])[:samples]
visualize_sequences(all_manufacturers[manufacturer][idx], all_manufacturers_fake[manufacturer][idx], titles=[f"{manufacturer} Real", "Fake"])

In [ ]:
manufacturer = manufacturers[1]
samples = 1000

idx = torch.randperm(all_manufacturers[manufacturer].shape[0])[:samples]
visualize_sequences(all_manufacturers[manufacturer][idx], all_manufacturers_fake[manufacturer][idx], titles=[f"{manufacturer} Real", "Fake"])

In [ ]:
df = compute_fid_all_types(all_manufacturers, all_manufacturers_fake, all_target, feature=2048)
df

In [ ]:
target_key = next(iter(all_target))
target = all_target[target_key]

target_identity = []
with torch.no_grad():
    for s in torch.split(target, 128):          # split() arg is chunk size, not count
        target_identity.append(model(s.cuda()).cpu())
target_identity = torch.cat(target_identity, dim=0)

# identity FID: G(target) vs target
metric = FrechetInceptionDistance(feature=2048).cuda()
_update_batched(metric, target,          real=True)
_update_batched(metric, target_identity, real=False)
identity_fid = metric.compute().item()
print("identity FID:", identity_fid)


fig = px.scatter(df, x="manufacturer", y="fid", color="type",
                 title="All vs. UNC GWH FID (real source vs. CUT fake, against target)")
fig.add_hline(y=identity_fid, line_dash="dash", line_color="green",
              annotation_text=f"identity G(target)→target = {identity_fid:.1f}",
              annotation_position="bottom right")
fig.show()


In [ ]:

# model_fn = "/mnt/GWH/Groups/FAMLI/Shared/C1_ML_Analysis/train_output/Cut/allvslast/allvsbutterfly/v0.1/epoch=90-val_loss=5.68.ckpt"
model_fn = '/mnt/GWH/Groups/FAMLI/Shared/C1_ML_Analysis/train_output/Cut/allvslast/allvsbutterfly/v0.2/epoch=37-val_loss_g_nce=3.03.ckpt'
model_bf = cut.CutG.load_from_checkpoint(model_fn).eval().cuda()

all_manufacturers_bf = all_target.copy()
for k in all_manufacturers.keys():
    if k != "Butterfly Network Inc":
        all_manufacturers_bf[k] = all_manufacturers[k]

all_manufacturers_bf_target = {"Butterfly Network Inc": all_manufacturers["Butterfly Network Inc"]}

all_manufacturers_bf_fake = {}
for manufacturer in all_manufacturers_bf:
    all_manufacturers_bf_fake[manufacturer] = []
    with torch.no_grad():
        splits = int(all_manufacturers_bf[manufacturer].shape[0]/128)
        for s in torch.split(all_manufacturers_bf[manufacturer], splits):
            s = s.cuda()
            s = model_bf(s)
            all_manufacturers_bf_fake[manufacturer].append(s.cpu())
        all_manufacturers_bf_fake[manufacturer] = torch.cat(all_manufacturers_bf_fake[manufacturer], dim=0).numpy()

df_bf = compute_fid_all_types(all_manufacturers_bf, all_manufacturers_bf_fake, all_manufacturers_bf_target, feature=2048)
df_bf




In [ ]:
target_key_bf = next(iter(all_manufacturers_bf_target))
target_bf = all_manufacturers_bf_target[target_key_bf]

target_identity = []
with torch.no_grad():
    for s in torch.split(target_bf, 128):          # split() arg is chunk size, not count
        target_identity.append(model_bf(s.cuda()).cpu())
target_identity = torch.cat(target_identity, dim=0)

# identity FID: G(target) vs target
metric = FrechetInceptionDistance(feature=2048).cuda()
_update_batched(metric, target_bf,          real=True)
_update_batched(metric, target_identity, real=False)
identity_fid = metric.compute().item()
print("identity FID:", identity_fid)


fig = px.scatter(df_bf, x="manufacturer", y="fid", color="type",
                 title="All vs. Butterfly FID (real source vs. CUT fake, against target)")
fig.add_hline(y=identity_fid, line_dash="dash", line_color="green",
              annotation_text=f"identity G(target)→target = {identity_fid:.1f}",
              annotation_position="bottom right")
fig.show()

In [ ]:
manufacturer = "UNC Global Women's Health"
samples = 1000

idx = torch.randperm(all_manufacturers_bf[manufacturer].shape[0])[:samples]
visualize_sequences(all_manufacturers_bf[manufacturer][idx], all_manufacturers_bf_fake[manufacturer][idx], titles=[f"{manufacturer} Real", "Fake"])